In [2]:
import time
import warnings
import numpy as np
import pandas as pd
import nashpy as nash
from scipy.optimize import fsolve

In [4]:
"""
Benchmark comparant trois méthodes de calcul d'équilibres de Nash
pour des jeux à 2 joueurs sous forme normale :

  1. Lemke-Howson (algorithme de pivotage, via nashpy)
  2. Support enumeration (énumération des supports, via nashpy)
  3. Homotopie (procédure de tracing logit / QRE, implémentation maison
     inspirée de Turocy 2005, car nashpy ne propose pas de méthode
     d'homotopie native)

Jeux testés :
  - Jeux classiques 2x2 : Battle of the Sexes, Chicken (Hawk-Dove),
    Prisoner's Dilemma
  - Jeux aléatoires de tailles variées (2x2, 3x3, 4x4, 5x5, 6x6)

Sorties :
  - results.csv            : résultats bruts (un run par ligne)
  - summary.csv             : temps moyen / taux de succès par méthode et taille
  - benchmark_plot.png      : graphique temps de calcul vs taille de jeu
"""

warnings.filterwarnings("ignore")
rng_global = np.random.default_rng(42)

# --------------------------------------------------------------------------
# 1. Définition des jeux
# --------------------------------------------------------------------------

def classic_games():
    """Retourne un dict {nom: (A, B)} de jeux classiques 2x2."""
    games = {}

    # Battle of the Sexes : deux équilibres purs + un mixte
    games["Battle of the Sexes"] = (
        np.array([[3, 0], [0, 2]]),
        np.array([[2, 0], [0, 3]]),
    )

    # Chicken / Hawk-Dove : deux équilibres purs + un mixte
    games["Chicken"] = (
        np.array([[0, 7], [2, 6]]),
        np.array([[0, 2], [7, 6]]),
    )

    # Prisoner's Dilemma : un unique équilibre pur (dominant)
    games["Prisoner's Dilemma"] = (
        np.array([[3, 0], [5, 1]]),
        np.array([[3, 5], [0, 1]]),
    )

    # Matching Pennies : un unique équilibre mixte (jeu à somme nulle)
    games["Matching Pennies"] = (
        np.array([[1, -1], [-1, 1]]),
        np.array([[-1, 1], [1, -1]]),
    )

    return games


def random_game(size, rng, payoff_range=(0, 10)):
    """Génère un jeu aléatoire (A, B) de dimension size x size,
    payoffs tirés uniformément dans payoff_range."""
    lo, hi = payoff_range
    A = rng.uniform(lo, hi, size=(size, size))
    B = rng.uniform(lo, hi, size=(size, size))
    return A, B


def random_games_battery(sizes=(2, 3, 4, 5, 6), n_per_size=8, seed=42):
    """Génère une batterie de jeux aléatoires pour plusieurs tailles."""
    rng = np.random.default_rng(seed)
    battery = []
    for s in sizes:
        for i in range(n_per_size):
            A, B = random_game(s, rng)
            battery.append({"name": f"random_{s}x{s}_#{i}", "size": s, "A": A, "B": B})
    return battery


# --------------------------------------------------------------------------
# 2. Les trois solveurs
# --------------------------------------------------------------------------

def solve_lemke_howson(A, B, timeout=None):
    """Lemke-Howson via nashpy : renvoie le premier équilibre trouvé
    (algorithme de pivotage, un seul équilibre par run côté typique)."""
    game = nash.Game(A, B)
    eq = game.lemke_howson(initial_dropped_label=0)
    sigma_r, sigma_c = eq
    # nashpy peut renvoyer un vecteur non normalisé en cas de dégénérescence
    if not (np.isclose(sigma_r.sum(), 1) and np.isclose(sigma_c.sum(), 1)):
        raise ValueError("Lemke-Howson: solution dégénérée / non normalisée")
    return [(sigma_r, sigma_c)]


def solve_support_enumeration(A, B):
    """Support enumeration via nashpy : renvoie TOUS les équilibres trouvés
    (complexité combinatoire, exhaustif mais coûteux)."""
    game = nash.Game(A, B)
    equilibria = list(game.support_enumeration())
    if not equilibria:
        raise ValueError("Support enumeration: aucun équilibre trouvé")
    return equilibria


def solve_homotopy(A, B, n_steps=200, tol=1e-8):
    """Procédure de tracing logit (homotopie), simplifiée à partir de
    Turocy (2005) 'A logit tracing procedure ...'.

    Idée : à t=0, la QLRE (quantal-response equilibrium) logit avec une
    température infinie (lambda=0) correspond à la stratégie uniforme,
    qui est l'unique solution. On fait croître lambda de 0 à +inf (on
    reparamètre t in [0,1] avec lambda = t/(1-t) pour t->1), et on suit
    la branche par continuation de Newton (fsolve), en utilisant la
    solution précédente comme point de départ (warm start). Quand
    lambda -> inf, la QLRE logit converge (génériquement) vers un
    équilibre de Nash du jeu.
    """
    m, n = A.shape

    def logit_system(x, lam):
        # x = [sigma_r (m), sigma_c (n)] sur le simplexe, paramétrés
        # directement (on résout avec contrainte via une variable de
        # normalisation implicite : on résout sur les probabilités et
        # on force la somme à 1 par une équation supplémentaire, en
        # remplaçant une composante).
        sigma_r = x[:m]
        sigma_c = x[m:]
        # gains espérés par action pure
        u_r = A @ sigma_c            # gain du joueur ligne pour chaque action
        u_c = sigma_r @ B            # gain du joueur colonne pour chaque action

        # équations du point fixe logit : sigma_i = softmax(lam * u_i)
        eq_r = sigma_r - _softmax(lam * u_r)
        eq_c = sigma_c - _softmax(lam * u_c)
        return np.concatenate([eq_r, eq_c])

    def _softmax(v):
        v = v - np.max(v)
        e = np.exp(v)
        return e / e.sum()

    # point de départ : lambda=0 -> stratégies uniformes (unique solution)
    x = np.concatenate([np.full(m, 1 / m), np.full(n, 1 / n)])

    # on fait croître lambda géométriquement de 0.1 à une valeur très grande :
    # cela concentre naturellement les pas là où la branche varie le plus
    # (proche de lambda -> infini, i.e. proche de l'équilibre de Nash limite),
    # tout en gardant une précision numérique élevée (xtol serré).
    lambdas = np.geomspace(0.1, 1e6, n_steps)
    for lam in lambdas:
        x_new, info, ier, msg = fsolve(
            logit_system, x, args=(lam,), full_output=True, xtol=1e-13
        )
        if ier != 1:
            # échec de convergence à ce pas : on retente avec plus d'itérations
            x_new, info, ier, msg = fsolve(
                logit_system, x, args=(lam,), full_output=True, xtol=1e-13, maxfev=5000
            )
            if ier != 1:
                continue  # on garde le dernier x valide et on poursuit la branche
        x = x_new

    sigma_r = np.clip(x[:m], 0, None)
    sigma_c = np.clip(x[m:], 0, None)
    sigma_r = sigma_r / sigma_r.sum()
    sigma_c = sigma_c / sigma_c.sum()

    # Vérification grossière que c'est bien proche d'un équilibre de Nash
    # (best-response check, tolérance tol)
    br_r = A @ sigma_c
    br_c = sigma_r @ B
    support_r = sigma_r > 1e-3
    support_c = sigma_c > 1e-3
    if support_r.any() and (br_r[support_r].max() - br_r.max()) < -1e-3:
        pass  # tolérance : on ne bloque pas le benchmark sur cette vérif
    return [(sigma_r, sigma_c)]


# --------------------------------------------------------------------------
# 3. Vérification qu'une paire de stratégies est un équilibre de Nash
# --------------------------------------------------------------------------

def is_nash_equilibrium(A, B, sigma_r, sigma_c, atol=1e-3):
    """Vérifie (à tolérance près) qu'aucun joueur n'a intérêt à dévier
    unilatéralement vers une action pure."""
    u_r = A @ sigma_c
    u_c = sigma_r @ B
    payoff_r = sigma_r @ u_r
    payoff_c = sigma_c @ u_c
    ok_r = u_r.max() <= payoff_r + atol
    ok_c = u_c.max() <= payoff_c + atol
    return bool(ok_r and ok_c)


# --------------------------------------------------------------------------
# 4. Boucle de benchmark
# --------------------------------------------------------------------------

METHODS = {
    "Lemke-Howson": solve_lemke_howson,
    "Support Enumeration": solve_support_enumeration,
    "Homotopy (logit tracing)": solve_homotopy,
}


def run_benchmark():
    rows = []

    # --- jeux classiques ---
    for name, (A, B) in classic_games().items():
        for method_name, solver in METHODS.items():
            row = _run_single(method_name, solver, A, B, name, size=A.shape[0])
            rows.append(row)

    # --- jeux aléatoires ---
    battery = random_games_battery(sizes=(2, 3, 4, 5, 6), n_per_size=8)
    for g in battery:
        for method_name, solver in METHODS.items():
            row = _run_single(method_name, solver, g["A"], g["B"], g["name"], size=g["size"])
            rows.append(row)

    return pd.DataFrame(rows)


def _run_single(method_name, solver, A, B, game_name, size):
    t0 = time.perf_counter()
    success = False
    n_eq = 0
    valid_eq = 0
    error_msg = ""
    try:
        equilibria = solver(A, B)
        n_eq = len(equilibria)
        for sigma_r, sigma_c in equilibria:
            if is_nash_equilibrium(A, B, sigma_r, sigma_c):
                valid_eq += 1
        success = valid_eq > 0
    except Exception as e:
        error_msg = str(e)[:120]
    elapsed = time.perf_counter() - t0

    return {
        "game": game_name,
        "size": size,
        "method": method_name,
        "time_s": elapsed,
        "success": success,
        "n_equilibria_returned": n_eq,
        "n_equilibria_valid": valid_eq,
        "error": error_msg,
    }


# --------------------------------------------------------------------------
# 5. Résumé + graphique
# --------------------------------------------------------------------------

def summarize(df):
    summary = (
        df.groupby(["method", "size"])
        .agg(
            mean_time_s=("time_s", "mean"),
            std_time_s=("time_s", "std"),
            success_rate=("success", "mean"),
            n_runs=("time_s", "count"),
        )
        .reset_index()
        .sort_values(["method", "size"])
    )
    return summary


def plot_results(summary, out_path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8, 5))
    for method, sub in summary.groupby("method"):
        # on ne trace que les tailles issues des jeux aléatoires (size >= 2, plusieurs points)
        sub = sub.sort_values("size")
        ax.plot(sub["size"], sub["mean_time_s"], marker="o", label=method)
    ax.set_xlabel("Taille du jeu (n x n)")
    ax.set_ylabel("Temps moyen (s)")
    ax.set_yscale("log")
    ax.set_title("Temps de calcul moyen par méthode et taille de jeu")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)


print("Lancement du benchmark...")
df = run_benchmark()
df.to_csv("results.csv", index=False)

summary = summarize(df)
summary.to_csv("summary.csv", index=False)

plot_results(summary, "benchmark_plot.png")

print("\n=== Résultats sur les jeux classiques ===")
classic_names = list(classic_games().keys())
print(df[df["game"].isin(classic_names)][
    ["game", "method", "time_s", "success", "n_equilibria_returned", "n_equilibria_valid"]
].to_string(index=False))

print("\n=== Résumé par méthode et taille (jeux aléatoires inclus) ===")
print(summary.to_string(index=False))

print("\nFichiers générés : results.csv, summary.csv, benchmark_plot.png")

Lancement du benchmark...

=== Résultats sur les jeux classiques ===
               game                   method   time_s  success  n_equilibria_returned  n_equilibria_valid
Battle of the Sexes             Lemke-Howson 0.001037     True                      1                   1
Battle of the Sexes      Support Enumeration 0.001153     True                      3                   3
Battle of the Sexes Homotopy (logit tracing) 0.066243     True                      1                   1
            Chicken             Lemke-Howson 0.000406     True                      1                   1
            Chicken      Support Enumeration 0.000940     True                      3                   3
            Chicken Homotopy (logit tracing) 0.073385     True                      1                   1
 Prisoner's Dilemma             Lemke-Howson 0.000443     True                      1                   1
 Prisoner's Dilemma      Support Enumeration 0.000954     True                     